In [2]:
# === 依赖与路径 ===
# 如环境已安装可注释掉下面两行
# %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# %pip install -q opencv-python numpy trimesh ipywidgets

import os
import sys
import glob
import gc
import time
import tempfile
import re
from pathlib import Path

import cv2
import torch
import numpy as np
import trimesh

PROJECT_ROOT = Path('/home/viplab/liukejia/project/vggt').resolve()
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / 'vggt'))
sys.path.append(str(PROJECT_ROOT.parent / 'reconstruct'))

print('PROJECT_ROOT =', PROJECT_ROOT)

from utils.visual_util import predictions_to_glb
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map
from utils.camera import parse_json_camera_extrinsic_intrinsic



PROJECT_ROOT = /home/viplab/liukejia/project/vggt


In [3]:
# === 参数配置 ===
input_path = '/home/viplab/liukejia/project/result/render_3d_stitched/'
output_path = '/home/viplab/liukejia/project/result/reconstruct_gt_scale4/'
depth_output_path = f"{input_path.rstrip('/')}_depth"

force_resize = True
conf_thres = 20.0
prediction_mode = 'Depthmap and Camera Branch'
save_npz = False
only_output_depth = False
save_raw_depth_npy = False

# camconf 相机配置
use_camconf_camera = True
camconf_dir = '/data/test_grid4_from_test_grid4/camconf'
camconf_angle_order = (0, 90, 180, 270)
strict_camconf = True

# 对齐模式：'umeyama'（推荐）或 'cam0'
CAMCONF_ALIGN_MODE = 'umeyama'

# 后处理：none | rigid | similarity | camera_fit
post_align_mode = 'none'
post_align_scale = 1.0
camera_fit_match_range = True
camera_fit_range_gain = 1.0
camera_fit_scale_min = 0.05
camera_fit_scale_max = 20.0
camera_fit_range_apply_to_camera = True

valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print('input_path =', input_path)
print('output_path =', output_path)
print('depth_output_path =', depth_output_path)
print('use_camconf_camera =', use_camconf_camera)
print('camconf_dir =', camconf_dir)
print('camconf_angle_order =', camconf_angle_order)
print('post_align_mode =', post_align_mode)
print('CAMCONF_ALIGN_MODE =', CAMCONF_ALIGN_MODE)

input_path = /home/viplab/liukejia/project/result/render_3d_stitched/
output_path = /home/viplab/liukejia/project/result/reconstruct_gt_scale4/
depth_output_path = /home/viplab/liukejia/project/result/render_3d_stitched_depth
use_camconf_camera = True
camconf_dir = /data/test_grid4_from_test_grid4/camconf
camconf_angle_order = (0, 90, 180, 270)
post_align_mode = none
CAMCONF_ALIGN_MODE = umeyama


In [5]:
# === 模型加载 ===
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)

model = VGGT()
_URL = 'https://huggingface.co/facebook/VGGT-1B/resolve/main/model.pt'
state_dict = torch.hub.load_state_dict_from_url(_URL, map_location='cpu')
model.load_state_dict(state_dict)
model.eval().to(device)

if torch.cuda.is_available():
    major_cc = torch.cuda.get_device_capability()[0]
    amp_dtype = torch.bfloat16 if major_cc >= 8 else torch.float16
else:
    amp_dtype = None
print('amp_dtype =', amp_dtype)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = cuda
amp_dtype = torch.bfloat16


In [6]:
# === 工具函数（精简整合）===
def collect_input_images(input_path: str, valid_exts=None):
    if valid_exts is None:
        valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    p = Path(input_path)
    if not p.exists():
        raise FileNotFoundError(f'input_path not found: {p}')
    if p.is_file():
        if p.suffix.lower() not in valid_exts:
            raise ValueError(f'Unsupported file extension: {p.suffix}')
        return [p]
    if p.is_dir():
        files = [x for x in sorted(p.iterdir()) if x.is_file() and x.suffix.lower() in valid_exts]
        if len(files) == 0:
            raise ValueError(f'No image files found in folder: {p}')
        return files
    raise ValueError(f'Invalid input_path: {p}')

def read_single_image_512(image_path: str, force_resize: bool = True):
    image_path = Path(image_path)
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f'Failed to read image: {image_path}')
    h, w = bgr.shape[:2]
    if (h, w) != (512, 512):
        if force_resize:
            bgr = cv2.resize(bgr, (512, 512), interpolation=cv2.INTER_AREA)
            print(f'[Info] resized {image_path.name}: {(w, h)} -> (512, 512)')
        else:
            raise ValueError(f'{image_path.name}: expect 512x512, got {(w, h)}')
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return rgb

def split_512_to_4x256_and_save(img_rgb: np.ndarray, target_dir: str):
    target_dir = Path(target_dir)
    images_dir = target_dir / 'images'
    images_dir.mkdir(parents=True, exist_ok=True)
    patches = [
        img_rgb[0:256, 0:256],
        img_rgb[0:256, 256:512],
        img_rgb[256:512, 0:256],
        img_rgb[256:512, 256:512],
    ]
    patch_paths = []
    for i, p in enumerate(patches):
        out_path = images_dir / f'{i:06d}.png'
        cv2.imwrite(str(out_path), cv2.cvtColor(p, cv2.COLOR_RGB2BGR))
        patch_paths.append(str(out_path))
    cv2.imwrite(str(target_dir / 'original_512.png'), cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    return patch_paths

def extract_camconf_base_stem(image_stem: str) -> str:
    m = re.match(r'^num123_([^_]+_[^_]+_[^_]+)_[^_]+$', image_stem)
    if m:
        return m.group(1)
    parts = image_stem.split('_')
    if len(parts) >= 5 and parts[0] == 'num123':
        return '_'.join(parts[1:4])
    if len(parts) >= 3:
        return '_'.join(parts[:3])
    raise ValueError(f'Cannot parse camconf stem from image stem: {image_stem}')

def _convert_extrinsic_shape_like(src_extr: np.ndarray, ref_extr: np.ndarray) -> np.ndarray:
    src = np.asarray(src_extr, dtype=np.float32)
    ref = np.asarray(ref_extr)
    if src.ndim != 3 or ref.ndim != 3:
        raise ValueError(f'extrinsic ndim error: src={src.shape}, ref={ref.shape}')
    if ref.shape[1:] == (3, 4):
        if src.shape[1:] == (3, 4):
            return src
        if src.shape[1:] == (4, 4):
            return src[:, :3, :4]
    if ref.shape[1:] == (4, 4):
        if src.shape[1:] == (4, 4):
            return src
        if src.shape[1:] == (3, 4):
            out = np.tile(np.eye(4, dtype=np.float32), (src.shape[0], 1, 1))
            out[:, :3, :4] = src
            return out
    raise ValueError(f'unsupported extrinsic shape: src={src.shape}, ref={ref.shape}')

def _camera_centers_from_extr(extrinsic: np.ndarray) -> np.ndarray:
    ext = np.asarray(extrinsic, dtype=np.float32)
    ext4 = _convert_extrinsic_shape_like(ext, np.zeros((ext.shape[0], 4, 4), dtype=np.float32))
    c2w = np.linalg.inv(ext4)
    return c2w[:, :3, 3]

def _estimate_similarity_umeyama(src: np.ndarray, dst: np.ndarray, with_scale: bool = True):
    src = np.asarray(src, dtype=np.float32)
    dst = np.asarray(dst, dtype=np.float32)
    if src.shape != dst.shape or src.ndim != 2 or src.shape[1] != 3:
        raise ValueError(f'bad shapes: src={src.shape}, dst={dst.shape}')
    mu_s = src.mean(axis=0)
    mu_d = dst.mean(axis=0)
    X = src - mu_s[None, :]
    Y = dst - mu_d[None, :]
    cov = (Y.T @ X) / src.shape[0]
    U, D, Vt = np.linalg.svd(cov)
    S = np.eye(3, dtype=np.float32)
    if np.linalg.det(U @ Vt) < 0:
        S[-1, -1] = -1.0
    R = (U @ S @ Vt).astype(np.float32)
    if with_scale:
        var_s = np.mean(np.sum(X * X, axis=1)) + 1e-8
        scale = float(np.sum(D * np.diag(S)) / var_s)
    else:
        scale = 1.0
    t = (mu_d - scale * (R @ mu_s)).astype(np.float32)
    return scale, R, t

def _apply_similarity_to_points_only(predictions: dict, s: float, R: np.ndarray, t: np.ndarray) -> dict:
    wp = np.asarray(predictions['world_points_from_depth'], dtype=np.float32)
    pts = wp.reshape(-1, 3)
    pts_new = (s * (pts @ R.T) + t[None, :]).astype(np.float32)
    predictions['world_points_from_depth'] = pts_new.reshape(wp.shape)
    return predictions

def _load_camconf_all_cameras_for_image(image_path: Path, camconf_dir: str, angle_order=(0, 90, 180, 270), H: int = 256, W: int = 256):
    base_stem = extract_camconf_base_stem(image_path.stem)
    exts, intrs, paths, valid_angles = [], [], [], []
    for a in angle_order:
        p = Path(camconf_dir) / f'{base_stem}_{int(a)}_camconf.json'
        if not p.exists():
            continue
        ext44, intr33 = parse_json_camera_extrinsic_intrinsic(str(p), device='cpu', H=H, W=W, dtype=torch.float32, flip_y=True)
        exts.append(ext44.squeeze(0).detach().cpu().numpy().astype(np.float32))
        intrs.append(intr33.squeeze(0).detach().cpu().numpy().astype(np.float32))
        paths.append(str(p))
        valid_angles.append(int(a))
    if len(exts) == 0:
        raise FileNotFoundError(f'no camconf found for {image_path.name} in {camconf_dir}')
    return np.stack(exts, axis=0), np.stack(intrs, axis=0), paths, valid_angles

In [8]:
# === 核心流程（内嵌补丁）===
def run_model(target_dir: str, model, image_path: Path = None, use_camconf_camera: bool = False, camconf_dir: str = None, camconf_angle_order=(0, 90, 180, 270), strict_camconf: bool = True) -> dict:
    target_dir = Path(target_dir)
    image_names = sorted(glob.glob(str(target_dir / 'images' / '*')))
    if len(image_names) == 0:
        raise ValueError(f'No images found in {target_dir}')
    images = load_and_preprocess_images(image_names).to(device)
    with torch.no_grad():
        if torch.cuda.is_available():
            with torch.cuda.amp.autocast(dtype=amp_dtype):
                predictions = model(images)
        else:
            predictions = model(images)
    extrinsic, intrinsic = pose_encoding_to_extri_intri(predictions['pose_enc'], images.shape[-2:])
    predictions['extrinsic'] = extrinsic
    predictions['intrinsic'] = intrinsic
    for key in list(predictions.keys()):
        if isinstance(predictions[key], torch.Tensor):
            predictions[key] = predictions[key].detach().cpu().numpy().squeeze(0)
    predictions['pose_enc_list'] = None
    predictions['camera_source'] = 'prediction'
    # 原始预测点云
    predictions['world_points_from_depth'] = unproject_depth_map_to_point_map(predictions['depth'], predictions['extrinsic'], predictions['intrinsic'])
    if use_camconf_camera:
        try:
            if image_path is None or camconf_dir is None:
                raise ValueError('camconf 模式需要 image_path 与 camconf_dir')
            H, W = int(predictions['depth'].shape[1]), int(predictions['depth'].shape[2])
            n_pred = int(predictions['depth'].shape[0])
            file_ext, file_intr, file_jsons, valid_angles = _load_camconf_all_cameras_for_image(image_path, camconf_dir, camconf_angle_order, H=H, W=W)
            if file_ext.shape[0] < n_pred:
                raise ValueError(f'camconf views not enough: need={n_pred}, got={file_ext.shape[0]}')
            pred_extr_orig = np.asarray(predictions['extrinsic'], dtype=np.float32)
            file_ext_like = _convert_extrinsic_shape_like(file_ext[:n_pred], pred_extr_orig)
            # 点云对齐（内嵌补丁）
            if CAMCONF_ALIGN_MODE == 'cam0':
                # cam0 对齐（简化：只用中心做尺度）
                pred_centers = _camera_centers_from_extr(pred_extr_orig)
                ref_centers = _camera_centers_from_extr(file_ext_like)
                n = min(len(pred_centers), len(ref_centers))
                if n >= 2:
                    s, R, t = _estimate_similarity_umeyama(pred_centers[:n], ref_centers[:n], with_scale=True)
                else:
                    s, R, t = 1.0, np.eye(3, dtype=np.float32), np.zeros((3,), dtype=np.float32)
            else:
                pred_centers = _camera_centers_from_extr(pred_extr_orig)
                ref_centers = _camera_centers_from_extr(file_ext_like)
                n = min(len(pred_centers), len(ref_centers))
                s, R, t = _estimate_similarity_umeyama(pred_centers[:n], ref_centers[:n], with_scale=True)
            predictions = _apply_similarity_to_points_only(predictions, s=float(s), R=np.asarray(R, dtype=np.float32), t=np.asarray(t, dtype=np.float32))
            # 相机锁定为 camconf
            predictions['extrinsic'] = file_ext_like.astype(np.float32)
            predictions['intrinsic'] = file_intr[:n_pred].astype(np.float32)
            predictions['camera_source'] = 'camconf_exact_dense'
            predictions['camera_json_paths'] = np.array(file_jsons[:n_pred], dtype=object)
            print(f'[EXACT] {image_path.name} | align={CAMCONF_ALIGN_MODE} | scale={float(s):.4f}')
        except Exception as e:
            if strict_camconf:
                raise
            print(f'[Warn] camconf failed, fallback prediction: {e}')
    return predictions

def _normalize_depth_for_save(depth: np.ndarray) -> np.ndarray:
    d = np.asarray(depth, dtype=np.float32)
    if d.ndim == 2:
        d = d[None, ...]
    elif d.ndim == 3:
        pass
    elif d.ndim == 4 and d.shape[-1] == 1:
        d = d[..., 0]
    elif d.ndim == 4 and d.shape[0] == 1:
        d = d[0]
    else:
        raise ValueError(f'Unexpected depth shape: {d.shape}')
    return d.astype(np.float32, copy=False)

def save_raw_depth(predictions: dict, depth_output_dir: str, output_stem: str):
    depth_dir = Path(depth_output_dir)
    depth_dir.mkdir(parents=True, exist_ok=True)
    depth = _normalize_depth_for_save(predictions['depth'])
    raw_npy_path = depth_dir / f'{output_stem}_depth.npy'
    np.save(raw_npy_path, depth)
    return {'depth_npy': str(raw_npy_path), 'depth_shape': tuple(depth.shape)}

def export_npz_and_glb(sample_target_dir: str, predictions: dict, output_dir: str, output_stem: str, conf_thres: float = 50.0, prediction_mode: str = 'Depthmap and Camera Branch', save_npz: bool = True, save_raw_depth_npy: bool = True, depth_output_dir: str = None, only_output_depth: bool = False):
    sample_target_dir = Path(sample_target_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    glb_path = output_dir / f'{output_stem}.glb'
    npz_path = output_dir / f'{output_stem}.npz'
    depth_outputs = None
    if save_raw_depth_npy:
        if depth_output_dir is None:
            depth_output_dir = f"{str(sample_target_dir).rstrip('/')}_depth"
        depth_outputs = save_raw_depth(predictions, depth_output_dir, output_stem)
    if only_output_depth:
        return None, None, depth_outputs
    if save_npz:
        np.savez(npz_path, **predictions)
    glbscene = predictions_to_glb(predictions, conf_thres=conf_thres, filter_by_frames='All', show_cam=True, mask_sky=False, mask_black_bg=False, mask_white_bg=False, target_dir=str(sample_target_dir), prediction_mode=prediction_mode)
    glbscene.export(file_obj=str(glb_path))
    return (str(npz_path) if save_npz else None), str(glb_path), depth_outputs

def reconstruct_one_image(image_path: Path, output_dir: Path, conf_thres: float, prediction_mode: str, force_resize: bool, save_npz: bool, save_raw_depth_npy: bool = True, depth_output_dir: str = None, only_output_depth: bool = False, use_camconf_camera: bool = False, camconf_dir: str = None, camconf_angle_order=(0, 90, 180, 270), strict_camconf: bool = True):
    img_rgb = read_single_image_512(str(image_path), force_resize=force_resize)
    sample_target_dir = Path(tempfile.mkdtemp(prefix=f'vggt_{image_path.stem}_'))
    _ = split_512_to_4x256_and_save(img_rgb, sample_target_dir)
    predictions = run_model(sample_target_dir, model, image_path=image_path, use_camconf_camera=use_camconf_camera, camconf_dir=camconf_dir, camconf_angle_order=camconf_angle_order, strict_camconf=strict_camconf)
    npz_path, glb_path, depth_outputs = export_npz_and_glb(sample_target_dir, predictions, output_dir, image_path.stem, conf_thres=conf_thres, prediction_mode=prediction_mode, save_npz=save_npz, save_raw_depth_npy=save_raw_depth_npy, depth_output_dir=depth_output_dir, only_output_depth=only_output_depth)
    return {'input_image': str(image_path), 'target_dir': str(sample_target_dir), 'glb_path': glb_path, 'npz_path': npz_path, 'depth_outputs': depth_outputs, 'camera_source': predictions.get('camera_source', 'prediction')}

In [9]:
# === 批量执行 ===
def reconstruct_from_input_path(input_path: str, output_path: str, conf_thres: float = 50.0, prediction_mode: str = 'Depthmap and Camera Branch', force_resize: bool = True, save_npz: bool = True, save_raw_depth_npy: bool = True, depth_output_dir: str = None, only_output_depth: bool = False, valid_exts=None, use_camconf_camera: bool = False, camconf_dir: str = None, camconf_angle_order=(0, 90, 180, 270), strict_camconf: bool = True):
    t0 = time.time()
    output_dir = Path(output_path)
    output_dir.mkdir(parents=True, exist_ok=True)
    if depth_output_dir is None:
        depth_output_dir = f"{str(input_path).rstrip('/')}_depth"
    Path(depth_output_dir).mkdir(parents=True, exist_ok=True)
    images = collect_input_images(input_path, valid_exts=valid_exts)
    print(f'[Start] total images: {len(images)}')
    results = []
    for i, img_path in enumerate(images, 1):
        print(f'[{i}/{len(images)}] processing: {img_path.name}')
        item = reconstruct_one_image(img_path, output_dir, conf_thres=conf_thres, prediction_mode=prediction_mode, force_resize=force_resize, save_npz=save_npz, save_raw_depth_npy=save_raw_depth_npy, depth_output_dir=depth_output_dir, only_output_depth=only_output_depth, use_camconf_camera=use_camconf_camera, camconf_dir=camconf_dir, camconf_angle_order=camconf_angle_order, strict_camconf=strict_camconf)
        results.append(item)
    elapsed = time.time() - t0
    print(f'[Done] {len(results)} image(s), elapsed: {elapsed:.2f}s')
    print('output_dir =', output_dir)
    return results

results = reconstruct_from_input_path(input_path=input_path, output_path=output_path, conf_thres=conf_thres, prediction_mode=prediction_mode, force_resize=force_resize, save_npz=save_npz, save_raw_depth_npy=save_raw_depth_npy, depth_output_dir=depth_output_path, only_output_depth=only_output_depth, valid_exts=valid_exts, use_camconf_camera=use_camconf_camera, camconf_dir=camconf_dir, camconf_angle_order=camconf_angle_order, strict_camconf=strict_camconf)

print('\n生成结果（前10条）:')
for row in results[:10]:
    glb_name = Path(row['glb_path']).name if row.get('glb_path', None) else 'SKIPPED'
    print(f"- {Path(row['input_image']).name} -> {glb_name} (camera={row.get('camera_source', 'prediction')})")

[Start] total images: 9
[1/9] processing: num123_-800_100_0_0007.png
[EXACT] num123_-800_100_0_0007.png | align=umeyama | scale=196.1247
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[2/9] processing: num123_-800_150_0_0008.png
[EXACT] num123_-800_150_0_0008.png | align=umeyama | scale=200.2509
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[3/9] processing: num123_-800_200_0_0009.png
[EXACT] num123_-800_200_0_0009.png | align=umeyama | scale=207.1613
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[4/9] processing: num123_-850_100_0_0004.png
[EXACT] num123_-850_100_0_0004.png | align=umeyama | scale=202.8000
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[5/9] processing: num123_-850_150_0_0005.png
[EXACT] num123_-850_150_0_0005.png | align=umeyama | scale=201.0079
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
[6/9] processing: num123_-850_200_0_0006.png
[EXACT] num123_-850_200_0_

In [10]:
# === 单个 GLB 的点云中心 + camconf 相机中心（核对）===
glb_path = '/home/viplab/liukejia/project/result/reconstruct_gt_scale3/num123_-800_100_0_0007.glb'
scene_or_mesh = trimesh.load(str(glb_path), force='scene')
pts_list = []
if isinstance(scene_or_mesh, trimesh.Scene):
    for g in scene_or_mesh.geometry.values():
        if hasattr(g, 'vertices'):
            v = np.asarray(g.vertices)
            if v.ndim == 2 and v.shape[1] == 3 and len(v) > 0:
                pts_list.append(v)
else:
    v = np.asarray(scene_or_mesh.vertices)
    if v.ndim == 2 and v.shape[1] == 3 and len(v) > 0:
        pts_list.append(v)
if len(pts_list) == 0:
    raise RuntimeError(f'GLB 内没有点云: {glb_path}')
pts = np.concatenate(pts_list, axis=0)
print('[GLB] point_center_mean =', pts.mean(axis=0))
print('[GLB] point_center_median =', np.median(pts, axis=0))
print('[GLB] point_count =', int(pts.shape[0]))
image_stem = Path(glb_path).stem
base_stem = extract_camconf_base_stem(image_stem)
centers = []
used = []
for a in camconf_angle_order:
    p = Path(camconf_dir) / f'{base_stem}_{int(a)}_camconf.json'
    if not p.exists():
        continue
    ext44, _ = parse_json_camera_extrinsic_intrinsic(str(p), device='cpu', H=256, W=256, dtype=torch.float32, flip_y=True)
    c = _camera_centers_from_extr(ext44.squeeze(0).detach().cpu().numpy()[None, ...])[0]
    centers.append(c.astype(np.float32))
    used.append(int(a))
if len(centers) == 0:
    raise RuntimeError(f'未找到 camconf: {base_stem} in {camconf_dir}')
centers = np.stack(centers, axis=0)
print('[camconf] used_angles =', used)
print('[camconf] camera_centers =')
for i, c in enumerate(centers):
    print(f'  cam[{i}] = ({c[0]:.4f}, {c[1]:.4f}, {c[2]:.4f})')
print('[camconf] center_mean =', centers.mean(axis=0))

[GLB] point_center_mean = [-802.15595031  103.65532539    9.682562  ]
[GLB] point_center_median = [-802.77832031  103.80348969   13.94436646]
[GLB] point_count = 858693
[camconf] used_angles = [0, 90, 180, 270]
[camconf] camera_centers =
  cam[0] = (-700.0000, 100.0000, 173.2051)
  cam[1] = (-800.0000, 200.0000, 173.2051)
  cam[2] = (-900.0000, 100.0000, 173.2051)
  cam[3] = (-800.0000, -0.0000, 173.2051)
[camconf] center_mean = [-800.        99.99999  173.20508]
